# Session 32: Logistic Regression Classification
## GitHub Deliverable
This notebook implements the Session 32 Logistic Regression baseline.

In [ ]:
import numpy as np
import pandas as pd
print('Session 32 notebook initialized.')

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Session 32: Logistic Regression Classification\n",
        "## GitHub Deliverable\n",
        "This notebook implements the Session 32 Logistic Regression baseline for student at-risk classification.\n",
        "\n",
        "### Classification target\n",
        "- `1` = at-risk student, where `G3 < 10`\n",
        "- `0` = successful student, where `G3 >= 10`\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1. Imports and repository configuration"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "import numpy as np\n",
        "import pandas as pd\n",
        "import matplotlib.pyplot as plt\n",
        "from IPython.display import display\n",
        "from sklearn.linear_model import LogisticRegression\n",
        "from sklearn.metrics import (\n",
        "    accuracy_score,\n",
        "    classification_report,\n",
        "    confusion_matrix,\n",
        "    ConfusionMatrixDisplay,\n",
        "    f1_score,\n",
        "    precision_score,\n",
        "    recall_score,\n",
        "    roc_auc_score,\n",
        ")\n",
        "from sklearn.model_selection import train_test_split\n",
        "from sklearn.pipeline import make_pipeline\n",
        "from sklearn.preprocessing import StandardScaler\n",
        "\n",
        "def find_repository_root(start_path=None):\n",
        "    start = Path(start_path or Path.cwd()).resolve()\n",
        "    candidates = [start, *start.parents]\n",
        "    for candidate in candidates:\n",
        "        if (candidate / \".git\").exists():\n",
        "            return candidate\n",
        "    raise FileNotFoundError(\"Unable to locate the Git repository root.\")\n",
        "\n",
        "REPO_ROOT = find_repository_root()\n",
        "DATA_DIRECTORY = REPO_ROOT / \"data\"\n",
        "PROCESSED_DIRECTORY = DATA_DIRECTORY / \"processed\"\n",
        "RAW_DIRECTORY = DATA_DIRECTORY / \"raw\"\n",
        "\n",
        "print(\"Repository root:\", REPO_ROOT)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2. Load the full-information modeling dataset"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "def read_table(path):\n",
        "    path = Path(path)\n",
        "    suffix = path.suffix.lower()\n",
        "    if suffix == \".csv\":\n",
        "        return pd.read_csv(path, sep=None, engine=\"python\")\n",
        "    if suffix == \".parquet\":\n",
        "        return pd.read_parquet(path)\n",
        "    if suffix in {\".pkl\", \".pickle\"}:\n",
        "        return pd.read_pickle(path)\n",
        "    raise ValueError(f\"Unsupported data-file type: {path}\")\n",
        "\n",
        "def first_existing_path(paths):\n",
        "    for path in paths:\n",
        "        if Path(path).exists():\n",
        "            return Path(path)\n",
        "    return None\n",
        "\n",
        "x_candidates = [\n",
        "    PROCESSED_DIRECTORY / \"X_full.parquet\",\n",
        "    PROCESSED_DIRECTORY / \"X_full.csv\",\n",
        "]\n",
        "y_candidates = [\n",
        "    PROCESSED_DIRECTORY / \"y_full.parquet\",\n",
        "    PROCESSED_DIRECTORY / \"y_full.csv\",\n",
        "]\n",
        "\n",
        "x_path = first_existing_path(x_candidates)\n",
        "y_path = first_existing_path(y_candidates)\n",
        "\n",
        "if x_path is not None and y_path is not None:\n",
        "    X_full = read_table(x_path)\n",
        "    y_loaded = read_table(y_path)\n",
        "    if isinstance(y_loaded, pd.DataFrame):\n",
        "        y = y_loaded[\"G3\"].copy() if \"G3\" in y_loaded.columns else y_loaded.iloc[:, 0].copy()\n",
        "    else:\n",
        "        y = pd.Series(y_loaded).copy()\n",
        "else:\n",
        "    raise FileNotFoundError(\"Required X_full and y_full dataset files not found.\")\n",
        "\n",
        "if \"G3\" in X_full.columns:\n",
        "    X_full = X_full.drop(columns=[\"G3\"])\n",
        "\n",
        "X_full.columns = X_full.columns.map(str)\n",
        "y = pd.Series(np.asarray(y).reshape(-1), name=\"G3\").reset_index(drop=True)\n",
        "X_full = X_full.reset_index(drop=True)\n",
        "print(\"Features shape:\", X_full.shape)\n",
        "print(\"Target length:\", len(y))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3. Prepare numeric model features"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "X_full = X_full.replace([np.inf, -np.inf], np.nan)\n",
        "numeric_columns = X_full.select_dtypes(include=\"number\").columns.tolist()\n",
        "categorical_columns = [col for col in X_full.columns if col not in numeric_columns]\n",
        "\n",
        "for col in numeric_columns:\n",
        "    X_full[col] = X_full[col].fillna(X_full[col].median() if not pd.isna(X_full[col].median()) else 0.0)\n",
        "\n",
        "for col in categorical_columns:\n",
        "    X_full[col] = X_full[col].astype(\"string\").fillna(\"Missing\")\n",
        "\n",
        "X_full = pd.get_dummies(X_full, columns=categorical_columns, drop_first=True, dtype=float)\n",
        "X_full = X_full.astype(float)\n",
        "print(\"Prepared feature matrix shape:\", X_full.shape)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4. Train/test split and class label definition"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.20, random_state=42)\n",
        "yc = (y < 10).astype(int)\n",
        "yctr = yc.loc[ytr.index].copy()\n",
        "ycte = yc.loc[yte.index].copy()\n",
        "\n",
        "print(\"Training target distribution:\\n\", yctr.value_counts())\n",
        "print(\"Test target distribution:\\n\", ycte.value_counts())"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 5. Fit Logistic Regression Pipeline and Evaluate"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "clf = make_pipeline(\n",
        "    StandardScaler(),\n",
        "    LogisticRegression(max_iter=1000, random_state=42)\n",
        ")\n",
        "clf.fit(Xtr_f, yctr)\n",
        "\n",
        "y_pred_logistic = clf.predict(Xte_f)\n",
        "y_proba_logistic = clf.predict_proba(Xte_f)[:, 1]\n",
        "\n",
        "def eval_clf(y_true, y_pred, y_proba):\n",
        "    return {\n",
        "        \"accuracy\": accuracy_score(y_true, y_pred),\n",
        "        \"precision\": precision_score(y_true, y_pred, zero_division=0),\n",
        "        \"recall\": recall_score(y_true, y_pred, zero_division=0),\n",
        "        \"f1\": f1_score(y_true, y_pred, zero_division=0),\n",
        "        \"roc_auc\": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) == 2 else np.nan,\n",
        "    }\n",
        "\n",
        "logistic_metrics = eval_clf(ycte, y_pred_logistic, y_proba_logistic)\n",
        "print(\"Logistic Regression Metrics:\", logistic_metrics)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 6. Final Validation & Completion Message"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "assert len(y_pred_logistic) == len(ycte)\n",
        "print(\"SESSION 32 GITHUB DELIVERABLE COMPLETED SUCCESSFULLY\")"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}